In [10]:

import pandas as pd

# ==========================================
# 1. LOAD THE ORIGINAL GOOGLE FORMS DATASET
# ==========================================

df = pd.read_csv("real_dataset.csv")

# Remove empty submissions without a participant code
df = df.dropna(subset=["Participant code"]).copy()
df = df.reset_index(drop=True)

# Preserve participant IDs as strings
df["Participant code"] = (
    df["Participant code"].astype(float).astype(int).astype(str)
)

# Create unique IDs for repeated participant codes.
# Keep the first occurrence unchanged.
df["_submission_order"] = range(len(df))

duplicate_number = (
    df.groupby("Participant code", sort=False).cumcount() + 1
)

duplicate_mask = duplicate_number > 1

df.loc[duplicate_mask, "Participant code"] += (
    "_" + duplicate_number[duplicate_mask].astype(str)
)

# ==========================================
# 2. DEFINE THE 13 SNIPPETS AND THEIR COLUMNS
# ==========================================

snippet_columns = {
    "S001": ("S001: How difficult is this code to understand?",
             "S001: How confident are you that you understood what this code does?"),

    "S002": ("S002: How difficult is this code to understand?",
             "S002: How confident are you that you understood what this code does?"),

    "S003": ("S003: How difficult is this code to understand?",
             "S003: How confident are you that you understood what this code does?"),

    "S004": ("S004: How difficult is this code to understand?",
             "S004: How confident are you that you understood what this code does?"),

    "S005": ("S005: How difficult is this code to understand?",
             "S005: How confident are you that you understood what this code does?"),

    "S006": ("S006: How difficult is this code to understand?",
             "S006: How confident are you that you understood what this code does?"),

    "S007": ("S007: How difficult is this code to understand?",
             "S007: How confident are you that you understood what this code does?"),

    "S008": ("S008: How difficult is this code to understand?",
             "S008: How confident are you that you understood what this code does?"),

    "S009": ("S009: How difficult is this code to understand?",
             "S009: How confident are you that you understood what this code does?"),

    "S010": ("S010: How difficult is this code to understand?",
             "S010: How confident are you that you understood what this code does?"),

    "S011": ("S011: How difficult is this code to understand?",
             "S011: How confident are you that you understood what this code does?"),

    "S012": ("S012: How difficult is this code to understand?",
             "S012: How confident are you that you understood what this code does?"),

    # The second S004 question is now S0013
    "S0013": (
        "S004: How difficult is this code to understand?.1",
        "S004: How confident are you that you understood what this code does?.1"
    )
}

# ==========================================
# 3. EXTRACT THE FEATURES OF EACH CODE SNIPPET
# ==========================================

features = {
    "S001":  [2, 0, 0, 0, 1, 3, 1],
    "S002":  [2, 0, 0, 0, 1, 2, 1],
    "S003":  [5, 1, 1, 0, 1, 4, 2],
    "S004":  [6, 2, 1, 1, 1, 4, 3],
    "S005":  [6, 2, 1, 1, 1, 4, 3],
    "S006":  [9, 4, 1, 3, 0, 3, 5],
    "S007":  [7, 2, 1, 2, 1, 5, 4],
    "S008":  [4, 1, 0, 1, 1, 2, 2],
    "S009":  [7, 3, 2, 1, 1, 8, 4],
    "S010":  [7, 3, 2, 1, 1, 7, 4],
    "S011":  [11, 3, 1, 2, 1, 7, 4],
    "S012":  [10, 3, 2, 1, 1, 12, 4],
    "S0013": [6, 2, 1, 1, 1, 5, 3]
}

feature_names = [
    "LOC",
    "nesting_depth",
    "loops",
    "conditions",
    "functions",
    "variables",
    "cyclomatic_complexity"
]

# ==========================================
# 4. CONVERT WIDE DATA INTO LONG FORMAT
# ==========================================

records = []

for _, row in df.iterrows():

    for snippet_id, columns in snippet_columns.items():

        difficulty_column, confidence_column = columns

        records.append({
            "participant_code": row["Participant code"],
            "_submission_order": row["_submission_order"],
            "form": "A",
            "snippet_id": snippet_id,

            "difficulty_1to10": row[difficulty_column],
            "confidence_1to5": row[confidence_column],

            "programming_experience":
                row["How long have you been programming?"],

            "python_comfort_1to5":
                row["How comfortable are you reading Python code?"]
        })

long_df = pd.DataFrame(records)

# ==========================================
# 5. ADD THE CODE FEATURES
# ==========================================

feature_df = pd.DataFrame.from_dict(
    features,
    orient="index",
    columns=feature_names
)

feature_df.index.name = "snippet_id"
feature_df = feature_df.reset_index()

long_df = long_df.merge(
    feature_df,
    on="snippet_id",
    how="left",
    validate="many_to_one"
)

# ==========================================
# 6. ENCODE PROGRAMMING EXPERIENCE
# ==========================================

experience_mapping = {
    "Less than 6 months": 0,
    "6 to 12 months": 1,
    "1 to 2 years": 2,
    "More than 2 years": 3
}

long_df["experience_level"] = (
    long_df["programming_experience"]
    .map(experience_mapping)
)

# ==========================================
# 7. SORT PARTICIPANTS AND SNIPPETS
# ==========================================

snippet_order = {
    snippet_id: number
    for number, snippet_id in enumerate(
        [
            "S001", "S002", "S003", "S004",
            "S005", "S006", "S007", "S008",
            "S009", "S010", "S011", "S012",
            "S0013"
        ],
        start=1
    )
}

long_df["_snippet_order"] = long_df["snippet_id"].map(
    snippet_order
)

long_df = long_df.sort_values(
    ["_submission_order", "_snippet_order"]
).reset_index(drop=True)

# ==========================================
# 8. KEEP THE REQUIRED 15 COLUMNS
# ==========================================

final_columns = [
    "participant_code",
    "form",
    "snippet_id",
    "difficulty_1to10",
    "confidence_1to5",
    "programming_experience",
    "python_comfort_1to5",
    "LOC",
    "nesting_depth",
    "loops",
    "conditions",
    "functions",
    "variables",
    "cyclomatic_complexity",
    "experience_level"
]

final_df = long_df[final_columns].copy()

# ==========================================
# 9. SAVE THE FINAL DATASET
# ==========================================

final_df.to_csv(
    "real_dataset_participant_snippet_sorted.csv",
    index=False
)

print("Dataset created successfully!")
print("Shape:", final_df.shape)
print("\nFirst 15 rows:")
print(final_df[["participant_code", "snippet_id"]].head(15))

# Validate the number of rows per submission
counts = final_df.groupby("participant_code")["snippet_id"].agg(
    ["count", "nunique"]
)

assert (counts["count"] == 13).all()
assert (counts["nunique"] == 13).all()

print("\nValidation successful!")
print("Each submission contains 13 unique snippets.")


Dataset created successfully!
Shape: (1040, 15)

First 15 rows:
   participant_code snippet_id
0              6623       S001
1              6623       S002
2              6623       S003
3              6623       S004
4              6623       S005
5              6623       S006
6              6623       S007
7              6623       S008
8              6623       S009
9              6623       S010
10             6623       S011
11             6623       S012
12             6623      S0013
13             2158       S001
14             2158       S002

Validation successful!
Each submission contains 13 unique snippets.
